# Olist ETL/ELT exercise — full pipeline

Generated from `notebooks/01_bronze_ingestion.py`, `02_etl_style_transform.py`, `03_question_choice_and_build.py` by `scripts/build_ipynb.py`. Edit those three files, not this one directly — re-run the script to regenerate.

Import this into Databricks (Workspace > Import) or run it against a cluster with Databricks Connect configured.

---
## 01 — Bronze ingestion

*(`01_bronze_ingestion.py`)*

# 01 — Bronze ingestion

Goal: land the raw Olist CSVs into your own Bronze Delta tables. Minimal
transformation here — just schema application and maybe a type cast.
No business logic yet, that comes later.

**Run the widget cell below first and set your schema name.**

In [ ]:
dbutils.widgets.text("schema", "", "Your schema (e.g. ra26_elt_ex_nils)")
dbutils.widgets.text("catalog", "dtr_sandbox", "Catalog")
dbutils.widgets.text("volume", "raw_data", "Volume (holds your CSVs)")

SCHEMA = dbutils.widgets.get("schema").strip()
CATALOG = dbutils.widgets.get("catalog").strip()
VOLUME = dbutils.widgets.get("volume").strip()
assert SCHEMA, "Set your schema in the widget above before running anything else."

# This is YOUR volume — the bundle provisioned it for you, and only you have
# access to it. Upload the Olist CSVs into it once (your teacher does this
# for everyone via scripts/upload_datasets.sh), then this path just works.
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# NOTE: Bronze tables live in the SAME schema as everything else you build
# (staging views, gold tables) — there's no separate bronze/gold schema
# split anymore. The "bronze_" prefix on the table name is what keeps things
# apart; that's why every table below is named bronze_<something>.
print(f"Bronze tables will land in {CATALOG}.{SCHEMA}, reading CSVs from {RAW_PATH}")

## Worked example: `bronze_customers`

This one's done for you — use it as the pattern for the rest.

In [ ]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/olist_customers_dataset.csv")
)

(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_customers")
)

display(spark.table(f"{CATALOG}.{SCHEMA}.bronze_customers").limit(5))

## Your turn: the other 7 tables

Same pattern every time: read CSV with header + inferSchema, write as a
managed Delta table under `{CATALOG}.{SCHEMA}`, named `bronze_<table>`.
Fill in `ingest_csv_to_bronze` below, then call it once per table.

While you do this, think about (you'll want an answer for the wrap-up
discussion): should any of these be partitioned? Does Bronze even need
partitioning, or does that decision belong further downstream? What
would you do differently for `order_items` (largest table) vs
`category_translation` (tiny lookup table)?

(There's a fully worked, tested version of this exact ingestion step in
`src/olist_pipeline/pipeline.py::ingest_bronze` if you want to compare
notes afterwards — don't peek before you've had a go.)

In [ ]:
def ingest_csv_to_bronze(csv_filename: str, table_name: str) -> None:
    """Read {RAW_PATH}/{csv_filename} and write it to {CATALOG}.{SCHEMA}.bronze_{table_name}.

    TODO: implement this using the same pattern as the customers example above.
    """
    # TODO: read the csv
    # TODO: write it as a managed delta table named bronze_{table_name}
    raise NotImplementedError("fill this in")


# TODO: call ingest_csv_to_bronze once for each of these
tables_to_ingest = [
    ("olist_orders_dataset.csv", "orders"),
    ("olist_order_items_dataset.csv", "order_items"),
    ("olist_order_payments_dataset.csv", "order_payments"),
    ("olist_order_reviews_dataset.csv", "order_reviews"),
    ("olist_products_dataset.csv", "products"),
    ("olist_sellers_dataset.csv", "sellers"),
    ("product_category_name_translation.csv", "category_translation"),
]

for csv_filename, table_name in tables_to_ingest:
    ingest_csv_to_bronze(csv_filename, table_name)
    print(f"landed bronze_{table_name}")

## Checkpoint

Before moving to `02_etl_style_transform.py`, confirm all 8 tables exist:

In [ ]:
expected = {f"bronze_{t}" for t in [
    "customers", "orders", "order_items", "order_payments",
    "order_reviews", "products", "sellers", "category_translation",
]}

existing = {r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()}
missing = expected - existing

if missing:
    print(f"Still missing: {sorted(missing)}")
else:
    print("All 8 Bronze tables are in place. On to the next notebook.")

---
## 02 — Question 1, the ETL way

*(`02_etl_style_transform.py`)*

# 02 — Question 1, the ETL way

Same business question you'll also answer in dbt
(`gold_delivery_performance_by_state.sql`): **delivery delay and lateness
rate by customer state.**

This time it's ETL-style: read the raw-ish Bronze tables, do ALL the
joining and aggregating here in PySpark, and only write the final,
already-business-ready result to Gold. Nothing gets loaded anywhere
queryable until the transform is done.

Required output columns (must match the dbt version exactly, so you can
compare them at the end):
- `customer_state`
- `order_count`
- `avg_delay_days`
- `pct_late`

In [ ]:
dbutils.widgets.text("schema", "", "Your schema (e.g. ra26_elt_ex_nils)")
dbutils.widgets.text("catalog", "dtr_sandbox", "Catalog")

SCHEMA = dbutils.widgets.get("schema").strip()
CATALOG = dbutils.widgets.get("catalog").strip()
assert SCHEMA, "Set your schema in the widget above."

# Everything lives in your one schema now — bronze_* tables in, this
# notebook's gold_*_etl output back into the same place.

In [ ]:
from pyspark.sql import functions as F

orders = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")
customers = spark.table(f"{CATALOG}.{SCHEMA}.bronze_customers")

## Build it

Steps:
1. Filter `orders` to `order_status == "delivered"` and
   `order_delivered_customer_date` not null
2. Join to `customers` on `customer_id`
3. Add a `delay_days` column: days between
   `order_delivered_customer_date` and `order_estimated_delivery_date`
   (positive = late). Look at `F.datediff`.
4. Group by `customer_state` and aggregate: count, avg delay, % late

In [ ]:
# TODO: step 1 — filter delivered orders with a non-null delivery date
delivered = orders  # placeholder, replace with your filter

# TODO: step 2 — join to customers
joined = delivered  # placeholder, replace with your join

# TODO: step 3 — add delay_days using F.datediff(delivered_col, estimated_col)
with_delay = joined.withColumn("delay_days", F.lit(None))  # placeholder

# TODO: step 4 — group by customer_state and compute order_count,
# avg_delay_days (rounded to 1 decimal), pct_late (rounded to 1 decimal)
result = (
    with_delay.groupBy("customer_state")
    .agg(
        F.count("*").alias("order_count"),
        F.lit(None).alias("avg_delay_days"),   # TODO
        F.lit(None).alias("pct_late"),          # TODO
    )
    .orderBy(F.desc("pct_late"))
)

display(result)

## Load — write the finished, business-ready table to Gold

Notice this is the ONLY write in this whole notebook. Everything before
it was in-memory transformation. That's the "T before L" of ETL.

In [ ]:
(
    result.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_delivery_performance_by_state_etl")
)

print(f"Written to {CATALOG}.{SCHEMA}.gold_delivery_performance_by_state_etl")

## Compare

Once you've also run the dbt model, compare the two results — they
should match exactly. If they don't, that's a good debugging exercise
in itself (which one do you trust, and why?).

There's also a THIRD version of this same question, written as a
tested Python function rather than notebook cells:
`src/olist_pipeline/transforms.py::delivery_performance_by_state`, with
its expected output asserted in `src/tests/test_transforms.py`. Once
you've built your own, it's worth reading that side by side and asking:
what would break first if the dataset changed? Which one would you
trust untouched for a year?

```sql
-- run this in a SQL cell/notebook once both exist:
SELECT 'etl' AS built_via, * FROM {catalog}.{schema}.gold_delivery_performance_by_state_etl
UNION ALL
SELECT 'elt' AS built_via, * FROM {catalog}.{schema}.gold_delivery_performance_by_state
ORDER BY customer_state, built_via
```

---
## 03 — Questions 2 & 3, your choice

*(`03_question_choice_and_build.py`)*

# 03 — Questions 2 & 3: your choice

For each question below, decide: PySpark (ETL — transform before load)
or dbt (ELT — transform after load)? There's no "correct" answer — the
point is to make the call and be able to defend it in the wrap-up.

If you pick **dbt** for a question, you don't need anything in this
notebook — go build the matching file in `dbt_project/models/marts/`.

If you pick **PySpark**, build it in the empty cells below.

In [ ]:
dbutils.widgets.text("schema", "", "Your schema (e.g. ra26_elt_ex_nils)")
dbutils.widgets.text("catalog", "dtr_sandbox", "Catalog")
SCHEMA = dbutils.widgets.get("schema").strip()
CATALOG = dbutils.widgets.get("catalog").strip()

## Question 2 — Monthly revenue by product category

`order_items` + `products` + `category_translation` + `orders`, revenue
by month and category (English name).

**Your choice:** PySpark / dbt  *(delete one)*

**Why:** _(one sentence — e.g. data volume, who'd maintain this in a real
team, how often it needs to run, how comfortable you are in each tool)_

In [ ]:
# If you picked PySpark for Q2, build it here.
# Required output columns: order_month, category_english, revenue, order_count
# (same shape as the dbt version in gold_monthly_revenue_by_category.sql)

# TODO (only if you picked PySpark): read bronze_order_items, bronze_products,
# bronze_category_translation, bronze_orders from {CATALOG}.{SCHEMA}; join;
# group by month + category; write to
# {CATALOG}.{SCHEMA}.gold_monthly_revenue_by_category_etl

## Question 3 — Does shipping speed drive satisfaction?

Review score bucketed by delivery delay.

**Your choice:** PySpark / dbt  *(delete one)*

**Why:** _(one sentence)_

In [ ]:
# If you picked PySpark for Q3, build it here.
# Required output columns: delay_bucket, order_count, avg_review_score
# (same shape as the dbt version in gold_review_score_vs_delivery_delay.sql)

# TODO (only if you picked PySpark): read bronze_orders, bronze_order_reviews
# from {CATALOG}.{SCHEMA}; compute delay_days and bucket it; join to reviews;
# group by bucket; write to
# {CATALOG}.{SCHEMA}.gold_review_score_vs_delay_etl

## Before the wrap-up discussion

Jot down (doesn't need to be more than a sentence each):
- Which approach did you pick for Q2 and Q3, and why?
- Now that you've built Q1 both ways — which did you personally find
  easier to write? Which would you rather **debug at 2am in production**?
- If this dataset were 100x bigger, would either of your Q2/Q3 choices change?
- `src/olist_pipeline/` builds all three questions as tested Python
  functions, not notebook cells — would you ship what you just wrote
  in this notebook to run unattended every night? What's missing?